# Get distal GeneScoreMatrix from ArchR project

In [1]:
here::i_am("revision/RNA_ATAC_integration_metacells/00_getGeneScoreMatrix_Distal.ipynb")

suppressPackageStartupMessages({
    library(data.table)
    library(ArchR)
    library(parallel)
})
addArchRVerbose(verbose = FALSE)

# Load default settings
source(here::here("settings.R"))
source(here::here("utils.R"))

here() starts at /rds/project/rds-SDzz0CATGms/users/bt392/atlasses/gastrulation_multiome/code


                                                   / |
                                                 /    \
            .                                  /      |.
            \\\                              /        |.
              \\\                          /           `|.
                \\\                      /              |.
                  \                    /                |\
                  \\#####\           /                  ||
                ==###########>      /                   ||
                 \\##==......\    /                     ||
            ______ =       =|__ /__                     ||      \\\
        ,--' ,----`-,__ ___/'  --,-`-===================##========>
       \               '        ##_______ _____ ,--,__,=##,__   ///
        ,    __==    ___,-,__,--'#'  ==='      `-'    | ##,-/
        -,____,---'       \\####\\________________,--\\_##,

In [2]:
###################
## Load settings ##
###################
args = list()
args$archr_directory <- file.path(io$basedir,"data/processed/atac/archR")
args$metadata <- file.path(io$basedir,"/results/atac/archR/qc/sample_metadata_after_qc.txt.gz")

In [3]:
##########################
## Load sample metadata ##
##########################

sample_metadata <- fread(args$metadata) %>%
  .[doublet_call==FALSE & pass_atacQC==TRUE & !sample %in% c('E8.5_CRISPR_T_KO','E8.5_CRISPR_T_WT')]

nrow(sample_metadata)

[1] 45887

In [4]:
########################
## Load ArchR Project ##
########################

setwd(args$archr_directory)

addArchRGenome("mm10")
addArchRThreads(threads = detectCores()-2) 

ArchRProject <- loadArchRProject(args$archr_directory)

# Subset to cells that pass QC
ArchRProject <- ArchRProject[sample_metadata$cell]

Setting default genome to Mm10.

Setting default number of Parallel threads to 30.

Successfully loaded ArchRProject!


                                                   / |
                                                 /    \
            .                                  /      |.
            \\\                              /        |.
              \\\                          /           `|.
                \\\                      /              |.
                  \                    /                |\
                  \\#####\           /                  ||
                ==###########>      /                   ||
                 \\##==......\    /                     ||
            ______ =       =|__ /__                     ||      \\\
        ,--' ,----`-,__ ___/'  --,-`-===================##========>
       \               '        ##_______ _____ ,--,__,=##,__   ///
        ,    __==    ___,-,__,--'#'  ==='      `-'    | ##,-/
        -,____,---'       \\####\\_

In [5]:
GeneScoreMatrix_distal.se = getMatrixFromProject(
  ArchRProj = ArchRProject,
  useMatrix = "GeneScoreMatrix_distal",
  useSeqnames = NULL,
  verbose = FALSE,
  binarize = FALSE,
  threads = getArchRThreads(),
  logFile = createLogFile("getMatrixFromProject")
)

In [8]:
rownames(GeneScoreMatrix_distal.se) = rowData(GeneScoreMatrix_distal.se)$name

In [12]:
assay(GeneScoreMatrix_distal.se[1:5, 1:5])

5 x 5 sparse Matrix of class "dgCMatrix"
       E8.5_rep1#GGCTGAGAGCTTAACA-1 E8.5_rep1#AGTAACACAACACTTG-1
Xkr4                          0.644                        0.596
Rp1                           0.441                        0.177
Sox17                         0.884                        0.366
Mrpl15                        0.242                        0.133
Lypla1                        0.093                        0.055
       E8.5_rep1#TATCCAGCACAGACTC-1 E8.5_rep1#TCCTCTAAGTCCTTCA-1
Xkr4                          0.505                        0.322
Rp1                           0.265                        0.169
Sox17                         1.036                        0.435
Mrpl15                        0.684                        0.352
Lypla1                        0.362                        0.120
       E8.5_rep1#GAGAACCAGACACTTA-1
Xkr4                          0.469
Rp1                           0.289
Sox17                         0.907
Mrpl15                        0.486

In [10]:
saveRDS(GeneScoreMatrix_distal.se, file.path(io$basedir, 'data/processed/atac/archR/Matrices/GeneScoreMatrix_distal_summarized_experiment.rds'))

In [ ]:
# Add imputation and log normalise in the same way as ArchR does it.

In [13]:
ArchRProject = addIterativeLSI(
                ArchRProj = ArchRProject,
                useMatrix = "PeakMatrix",
                name = "IterativeLSI",
                varFeatures = 50000,
                dimsToUse = 1:40)  

Checking Inputs...



In [18]:
ArchRProject = addImputeWeights(
                              ArchRProj = ArchRProject,
                              reducedDims = "IterativeLSI")

In [ ]:
ArchRProject@reducedDims$IterativeLSI$svd

In [22]:
GeneScoreMatrix_distal_imputed.se = imputeMatrix(
                                          mat = assay(GeneScoreMatrix_distal.se),
                                          imputeWeights = getImputeWeights(ArchRProject),
                                          threads = getArchRThreads(),
                                          verbose = FALSE,
                                          logFile = createLogFile("imputeMatrix")
)

Getting ImputeWeights



In [24]:
GeneScoreMatrix_distal_imputed.se[1:5, 1:5]

5 x 5 Matrix of class "dgeMatrix"
       E8.5_rep1#GGCTGAGAGCTTAACA-1 E8.5_rep1#AGTAACACAACACTTG-1
Xkr4                      0.4575237                    0.5999807
Rp1                       0.1671509                    0.1804393
Sox17                     0.6655203                    0.7693957
Mrpl15                    0.3087879                    0.2826654
Lypla1                    0.2577785                    0.2553686
       E8.5_rep1#TATCCAGCACAGACTC-1 E8.5_rep1#TCCTCTAAGTCCTTCA-1
Xkr4                      0.5852406                    0.4681305
Rp1                       0.1868824                    0.1747906
Sox17                     1.1951592                    0.8836580
Mrpl15                    0.5996500                    0.3263946
Lypla1                    0.5730072                    0.2634207
       E8.5_rep1#GAGAACCAGACACTTA-1
Xkr4                      0.4949356
Rp1                       0.2151606
Sox17                     0.9931715
Mrpl15                    0.3870210
Lypla1

In [20]:
# Log normalise imputed matrix
mat = assay(GeneScoreMatrix_distal_imputed.se)
mat <- log(mat + 1)

ERROR: Error in h(simpleError(msg, call)): error in evaluating the argument 'x' in selecting a method for function 'assay': object 'GeneScoreMatrix_distal_imputed.se' not found


In [ ]:
saveRDS(mat, file.path(io$basedir, 'data/processed/atac/archR/Matrices/GeneScoreMatrix_distal_summarized_experiment.rds'))